<a href="https://colab.research.google.com/github/JCF-the-creator/Multi-Motorsport-Analytics---Desempenho-e-Estrat-gia/blob/main/Codigo_PY_MotoGP/MotoGp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas tabula-py PyPDF2

import os
import requests
import pandas as pd
from pathlib import Path
import zipfile
import tabula
import shutil
from pathlib import Path

ROOT = Path("motogp")
ROOT.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}
YEARS = range(2002, 2027)

# Lista de circuitos (slugs oficiais usados nos PDFs)
CIRCUITS = ["VAL","CAT","BRNO","MIS","SEP","QAT","AUS","ARG","JPN","ITA","GER","NED","GBR","USA","POR","ARA","SMR","MAL","FRA"]

def baixar_pdf(ano, circuito):
    url = f"https://resources.motogp.com/files/results/{ano}/MotoGP/{circuito}/RAC/classification.pdf"
    pasta_base = ROOT / str(ano) / circuito
    pasta_base.mkdir(parents=True, exist_ok=True)
    pdf_path = pasta_base / "classification.pdf"

    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            with open(pdf_path, "wb") as f:
                f.write(r.content)
            print(f"[SAVED] {pdf_path}")
            return pdf_path
        else:
            print(f"[NO PDF] {url}")
    except Exception as e:
        print(f"[ERROR] {url}: {e}")
    return None

def converter_pdf(pdf_path):
    try:
        # Extrair tabelas do PDF usando tabula
        dfs = tabula.read_pdf(str(pdf_path), pages="all", multiple_tables=True)
        if dfs:
            df = dfs[0]  # pega a primeira tabela
            # salvar CSV e JSON
            csv_path = pdf_path.with_suffix(".csv")
            json_path = pdf_path.with_suffix(".json")
            df.to_csv(csv_path, index=False)
            df.to_json(json_path, orient="records", indent=4, force_ascii=False)
            print(f"[CONVERTED] {csv_path}, {json_path}")
    except Exception as e:
        print(f"[ERROR CONVERT] {pdf_path}: {e}")

def zipar_ano(ano):
    pasta_ano = ROOT / str(ano)
    if not pasta_ano.exists():
        return
    zip_path = ROOT / f"{ano}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in pasta_ano.rglob("*"):
            if file_path.is_file():
                arcname = file_path.relative_to(ROOT)
                zipf.write(file_path, arcname)
    print(f"[ZIP CREATED] {zip_path}")

# Loop principal
for ano in YEARS:
    for circuito in CIRCUITS:
        pdf_path = baixar_pdf(ano, circuito)
        if pdf_path:
            converter_pdf(pdf_path)
    zipar_ano(ano)

print("Concluído: PDFs + CSV + JSON organizados e ZIPs criados em motogp/")
ROOT = Path("motogp")
TOTAL = Path("motogp_total")
TOTAL.mkdir(parents=True, exist_ok=True)

# mover todos os zips para a pasta total
for zip_file in ROOT.glob("*.zip"):
    destino = TOTAL / zip_file.name
    shutil.move(str(zip_file), destino)
    print(f"[MOVED] {zip_file} -> {destino}")

print("Concluído: todos os ZIPs reunidos em motogp_total/")
